In [ ]:
# Quick check: does a plain requests.get to Spiegel still get a response at all right now,
# or does it also time out (which would confirm an IP-level block rather than a JS challenge)?

import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Encoding": "gzip",
}

try:
    r = requests.get("https://www.spiegel.de/sitemap.xml", headers=HEADERS, timeout=15)
    print("status:", r.status_code)
    print(r.text[:300])
except Exception as e:
    print("EXCEPTION:", repr(e))

In [ ]:
# Try fetching the Spiegel sitemap through cloudscraper instead of curl_cffi.
# Same monthly-shard counting logic as before.

import re
import gzip
import cloudscraper

scraper = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "windows", "mobile": False})

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

def fetch_xml(url, is_gz=False):
    try:
        r = scraper.get(url, headers=HEADERS, timeout=30)
        print(f"  status={r.status_code} url={url}")
        if r.status_code != 200:
            print("  preview:", r.text[:300])
            return None
        content = gzip.decompress(r.content) if is_gz else r.content
        return content.decode("utf-8", errors="ignore")
    except Exception as e:
        print("  EXCEPTION:", repr(e))
        return None

main_xml = fetch_xml("https://www.spiegel.de/sitemap.xml")
if main_xml:
    print(main_xml[:1000])

## Polityka missing articles

In [ ]:
# Check which Polityka sitemap shard(s) (article0.xml through article7.xml) actually contain
# February-March 2022 articles, by sampling published_time from each shard — same method you
# used in your notebook (cell 52), just applied to ALL 8 shards instead of just one.

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

def get_urls(shard_url):
    r = requests.get(shard_url, timeout=60)
    soup = BeautifulSoup(r.text, "xml")
    return [loc.text for loc in soup.find_all("loc")]

def sample_years(urls, sample_size=100):
    sample = pd.Series(urls).sample(min(sample_size, len(urls)), random_state=42)
    years = []
    for url in tqdm(sample, leave=False):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            soup = BeautifulSoup(r.text, "lxml")
            meta = soup.find("meta", attrs={"property": "article:published_time"})
            if meta:
                years.append(meta["content"][:7])  # YYYY-MM
        except Exception:
            pass
    return years

for i in range(8):
    shard_url = f"https://www.polityka.pl/sitemap/article{i}.xml"
    urls = get_urls(shard_url)
    years = sample_years(urls)
    print(f"\narticle{i}.xml — {len(urls)} urls total")
    print(pd.Series(years).value_counts().sort_index().to_string())
    

In [ ]:
# Step 1: download the missing Feb-Apr 2022 Polityka articles from article5.xml —
# same category restriction, same article-parsing logic (og:title / article:published_time /
# <article><p> text), then filter by date window and by your actual title keyword list.

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

MAX_WORKERS = 10
START_DATE = pd.Timestamp("2022-02-20", tz="UTC")
END_DATE = pd.Timestamp("2022-04-30 23:59:59", tz="UTC")

target_categories = {
    "kraj", "swiat", "spoleczenstwo", "rynek", "historia",
    "temattygodnia", "rysunkisatyryczne", "batorywpolityce",
    "politykaohistorii", "informacjeprasowe", "aktualnosci",
    "polskiedemo", "pokolenia", "portretpolityka", "swiatnazakrecie",
    "passentpassent", "rakowski", "postnormalnosc", "wydawnictwa",
    "fotoreportaze", "tobylblad", "galerie", "redakcja",
}

russia_keywords_pl = re.compile(
    r"\b("
    r"rosj\w*|rosyjsk\w*|kreml\w*|putin\w*|moskw\w*|"
    r"ławrow\w*|lawrow\w*|pieskow\w*|szojgu\w*|szoigu\w*|shoigu\w*|"
    r"gierasimow\w*|miedwiediew\w*|zacharowa\w*|miszustin\w*|"
    r"gazprom\w*|rosnieft\w*|rosatom\w*|transnieft\w*|łukoil\w*|lukoil\w*|"
    r"donieck\w*|ługańsk\w*|luhansk\w*|krym\w*"
    r")\b",
    re.IGNORECASE,
)

# =====================================================
# HELPERS
# =====================================================

def get_category(url):
    parts = url.replace("https://www.polityka.pl/", "").split("/")
    return parts[1] if len(parts) >= 2 else None

def get_meta(soup, prop):
    tag = soup.find("meta", property=prop)
    return tag["content"].strip() if tag and tag.get("content") else None

def get_article_id(url):
    m = re.search(r"/(\d+),", url)
    return m.group(1) if m else None

def parse_article(row):
    url, category = row
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return {"url": url, "category": category, "error": f"status_{r.status_code}"}

        soup = BeautifulSoup(r.text, "html.parser")

        title = get_meta(soup, "og:title")
        publish_date = get_meta(soup, "article:published_time")

        article = soup.find("article")
        text = ""
        if article:
            paragraphs = [p.get_text(" ", strip=True) for p in article.find_all("p")]
            text = "\n".join(t for t in paragraphs if t)

        return {
            "article_id": get_article_id(url),
            "category": category,
            "url": url,
            "title": title,
            "publish_date": publish_date,
            "text": text,
            "text_length": len(text),
            "error": None,
        }
    except Exception as e:
        return {"url": url, "category": category, "error": str(e)}

# =====================================================
# LOAD SITEMAP AND FILTER BY CATEGORY
# =====================================================

r = requests.get("https://www.polityka.pl/sitemap/article5.xml", timeout=60)
soup = BeautifulSoup(r.text, "xml")
urls = [loc.text for loc in soup.find_all("loc")]
print(f"article5.xml total urls: {len(urls)}")

urls_with_category = [(u, get_category(u)) for u in urls]
urls_filtered = [(u, cat) for u, cat in urls_with_category if cat in target_categories]
print(f"in target categories: {len(urls_filtered)}")

# =====================================================
# FETCH FULL ARTICLES (title, date, text) FOR CANDIDATES
# =====================================================

rows = []
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(parse_article, pair) for pair in urls_filtered]
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        (failed if result.get("error") else rows).append(result)

df = pd.DataFrame(rows)
print(f"\nfetched ok: {len(df)}, failed: {len(failed)}")

# =====================================================
# FILTER: date window + Russia in title
# =====================================================

df["date_parsed"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True)
df_window = df[(df["date_parsed"] >= START_DATE) & (df["date_parsed"] <= END_DATE)].copy()
print(f"in 20 Feb - 30 Apr 2022 window: {len(df_window)}")

df_window["title_has_russia"] = df_window["title"].fillna("").str.contains(russia_keywords_pl, na=False)
df_ua_ru = df_window[df_window["title_has_russia"]].drop(columns=["title_has_russia", "date_parsed"])
print(f"with Russia in title: {len(df_ua_ru)}")

df_ua_ru = df_ua_ru.rename(columns={"publish_date": "publish_date"})  # keep same column name as your pipeline

df_ua_ru.to_csv("polityka_article5_feb_apr_2022_russia.csv", index=False, encoding="utf-8-sig")
if failed:
    pd.DataFrame(failed).to_csv("polityka_article5_failed.csv", index=False, encoding="utf-8-sig")

print("\nsaved: polityka_article5_feb_apr_2022_russia.csv")

In [ ]:
# Download images for the 205 recovered Feb-Apr 2022 Polityka articles, same logic as your
# original cell 67 (og:image + all <img> tags), then merge everything into a format ready
# to append to polityka_ukraine_russia_with_images.csv.

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

df_new = pd.read_csv("polityka_article5_feb_apr_2022_russia.csv")

rows = []
failed = []

for _, row in tqdm(df_new.iterrows(), total=len(df_new)):
    try:
        r = requests.get(row["url"], headers=HEADERS, timeout=30)
        if r.status_code != 200:
            failed.append({"url": row["url"], "error": f"status_{r.status_code}"})
            continue

        soup = BeautifulSoup(r.text, "lxml")
        images = set()

        og = soup.find("meta", property="og:image")
        if og:
            url = og.get("content")
            if url and url.startswith("http"):
                images.add(url)

        for img in soup.find_all("img"):
            url = img.get("src") or img.get("data-src") or img.get("data-original")
            if url and url.startswith("http"):
                images.add(url)

        for image_url in images:
            rows.append({"article_id": row["article_id"], "url": row["url"], "image_url": image_url})

    except Exception as e:
        failed.append({"url": row["url"], "error": str(e)})

df_images_new = pd.DataFrame(rows)
df_failed_new = pd.DataFrame(failed)

print(f"\nimages found: {df_images_new.shape}")
print(f"failed: {df_failed_new.shape}")

df_images_new.to_csv("polityka_article5_images.csv", index=False, encoding="utf-8-sig")
df_failed_new.to_csv("polityka_article5_images_failed.csv", index=False, encoding="utf-8-sig")

# ---------- combine images into pipe-separated column per article, same as your pipeline ----------
images_grouped_new = (
    df_images_new.groupby("article_id")["image_url"]
    .apply(lambda s: " | ".join(s.dropna().astype(str).unique()))
    .reset_index()
)

df_new_final = df_new.merge(images_grouped_new, on="article_id", how="left")
df_new_final.to_csv("polityka_article5_feb_apr_2022_with_images.csv", index=False, encoding="utf-8-sig")

print("\nsaved: polityka_article5_feb_apr_2022_with_images.csv")

In [ ]:
# Merge the recovered Feb-Apr 2022 Polityka articles into polityka_final_data.csv, and
# download their actual image files into polityka_final_images/ using filenames that continue
# your existing numbering scheme (so they don't collide with existing files).

import os
import re
import requests
import pandas as pd
from tqdm import tqdm

TARGET_COLUMNS = ["category", "article_url", "title", "date", "text", "image_url", "image_filename"]
IMAGES_DIR = "polityka_final_images"
os.makedirs(IMAGES_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137.0 Safari/537.36"
}

# =====================================================
# LOAD EXISTING DATA + NEW DATA
# =====================================================

df_existing = pd.read_csv("polityka_final_data.csv", low_memory=False)
df_new = pd.read_csv("polityka_article5_feb_apr_2022_with_images.csv")

# =====================================================
# FIND CURRENT MAX FILENAME NUMBER (to continue the sequence, not collide)
# =====================================================

existing_numbers = []
for value in df_existing["image_filename"].dropna():
    for name in str(value).split("|"):
        m = re.match(r"(\d+)\.\w+$", name.strip())
        if m:
            existing_numbers.append(int(m.group(1)))

next_number = (max(existing_numbers) + 1) if existing_numbers else 0
print(f"continuing filenames from: {next_number:06d}")

# =====================================================
# DOWNLOAD NEW IMAGES, ASSIGNING SEQUENTIAL FILENAMES
# =====================================================

def download_image(url, filepath):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30, stream=True)
        if r.status_code != 200:
            return False
        with open(filepath, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        return True
    except Exception:
        return False

new_image_filenames = []  # per-row pipe-separated filename string
counter = next_number

for _, row in tqdm(df_new.iterrows(), total=len(df_new)):
    urls = [u.strip() for u in str(row.get("image_url", "")).split("|") if u.strip() and u.strip() != "nan"]
    filenames = []

    for url in urls:
        ext = os.path.splitext(url.split("?")[0])[1]
        if ext.lower() not in (".jpg", ".jpeg", ".png", ".webp", ".gif"):
            ext = ".jpg"
        filename = f"{counter:06d}{ext}"
        filepath = os.path.join(IMAGES_DIR, filename)

        if download_image(url, filepath):
            filenames.append(filename)
            counter += 1

    new_image_filenames.append(" | ".join(filenames))

df_new["image_filename"] = new_image_filenames

print(f"\ndownloaded images: {counter - next_number}")

# =====================================================
# RENAME COLUMNS TO MATCH polityka_final_data.csv SCHEMA
# =====================================================

df_new_renamed = df_new.rename(columns={
    "url": "article_url",
    "publish_date": "date",
})

for col in TARGET_COLUMNS:
    if col not in df_new_renamed.columns:
        df_new_renamed[col] = pd.NA

df_new_renamed = df_new_renamed[TARGET_COLUMNS]

# =====================================================
# MERGE AND SAVE
# =====================================================

df_merged = pd.concat([df_existing, df_new_renamed], ignore_index=True)
df_merged = df_merged.drop_duplicates(subset=["article_url"])

print(f"\nbefore: {len(df_existing)}, added: {len(df_new_renamed)}, after dedup: {len(df_merged)}")

df_merged.to_csv("polityka_final_data.csv", index=False, encoding="utf-8-sig")
print("saved: polityka_final_data.csv")